# EXP-01: Logistic Regression — Phân tích kết quả

**TV1 — EXP-01 (TF-IDF + Logistic Regression)**

Notebook này đọc kết quả từ MLflow artifacts và files JSON — **không chứa training logic**.

> **Trạng thái:** CHƯA THỰC HIỆN — chạy `python -m src.models.lr_pipeline` trước.

## Nội dung
1. Load kết quả 3 variants từ artifacts
2. Bảng so sánh Macro-F1 / per-class F1
3. Confusion matrix từng variant
4. Phân tích ablation: preprocessing, TF-IDF mode, class weighting
5. Kết luận EXP-01


In [ ]:
import sys, json
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.eda.eda_utils import plot_confusion_matrix, plot_f1_comparison

ARTIFACTS_BASE = ROOT / 'models' / 'baselines'
VARIANTS = ['A', 'B', 'C']


In [ ]:
# Load metrics từ artifact files
results = []
for v in VARIANTS:
    metrics_path = ARTIFACTS_BASE / f'EXP-01-{v}' / 'metrics.json'
    config_path = ARTIFACTS_BASE / f'EXP-01-{v}' / 'config.json'
    if not metrics_path.exists():
        print(f'CHƯA THỰC HIỆN: EXP-01-{v} chưa có kết quả ({metrics_path})')
        continue
    with open(metrics_path) as f:
        metrics = json.load(f)
    with open(config_path) as f:
        config = json.load(f)
    results.append({'variant': v, 'metrics': metrics, 'config': config})
    print(f'  EXP-01-{v}: Macro-F1={metrics["macro_f1"]:.4f}')

if not results:
    print('\nCHƯA THỰC HIỆN: Không có kết quả nào. Chạy: python -m src.models.lr_pipeline')


## 1. Bảng so sánh các biến thể

In [ ]:
if results:
    rows = []
    for r in results:
        m = r['metrics']
        pc = m['per_class']
        rows.append({
            'Variant': f"EXP-01-{r['variant']}",
            'Preprocessing': r['config']['preprocessing'],
            'TF-IDF mode': r['config']['tfidf_mode'],
            'C': r['config']['lr_C'],
            'class_weight': r['config']['lr_class_weight'],
            'Macro-F1': round(m['macro_f1'], 4),
            'CLEAN-F1': round(pc['CLEAN']['f1'], 4),
            'OFFENSIVE-F1': round(pc['OFFENSIVE']['f1'], 4),
            'HATE-F1': round(pc['HATE']['f1'], 4),
            'Train time (s)': m.get('train_time_s', 'N/A'),
        })
    df_cmp = pd.DataFrame(rows)
    display(df_cmp)
else:
    print('CHƯA THỰC HIỆN')


## 2. So sánh F1 theo lớp (bar chart)

In [ ]:
if results:
    fig = plot_f1_comparison(results)
    fig.savefig(ROOT / 'notebooks' / 'exp01_f1_comparison.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('CHƯA THỰC HIỆN')


## 3. Confusion Matrix — từng variant

In [ ]:
if results:
    for r in results:
        cm = r['metrics']['confusion_matrix']
        fig = plot_confusion_matrix(cm, title=f"EXP-01-{r['variant']} Confusion Matrix (normalized)")
        fig.savefig(
            ROOT / 'notebooks' / f"exp01_{r['variant']}_confusion.png",
            dpi=120, bbox_inches='tight'
        )
        plt.show()
else:
    print('CHƯA THỰC HIỆN')


## 4. Phân tích ablation

> **Điền sau khi có kết quả thực tế:**
>
> **Vấn đề baseline (A):** ...
>
> **Giả thuyết B:** Thêm basic preprocessing và char n-gram giúp bắt subword patterns → kết quả: ...
>
> **Giả thuyết C:** Teencode normalization + class_weight=balanced cải thiện HATE recall → kết quả: ...
>
> **Kết luận:** Variant X đạt Macro-F1 = ? trên validation; lý do chọn: ...


## 5. MLflow Run IDs


In [ ]:
for v in VARIANTS:
    run_id_path = ARTIFACTS_BASE / f'EXP-01-{v}' / 'run_id.txt'
    if run_id_path.exists():
        run_id = run_id_path.read_text().strip()
        print(f'EXP-01-{v}: run_id = {run_id}')
    else:
        print(f'EXP-01-{v}: CHƯA THỰC HIỆN')
